# Guide-outlier Jaccard

Outlier gRNAs are guides with `pval_outlier < 0.05` in each dataset's `targeting_outlier_table.csv`. Pairwise Jaccard between the resulting per-dataset sets — lighter = more shared outliers.

Note: Gersbach Hep's run does not return meaningful outlier p-values (all 1.0), so its row/column is all-zero. Flagging in the WG1 ED discussion.

**Input:** `results/guide_outlier_jaccard/guide_outlier_jaccard.tsv` (from `scripts/compute_guide_outlier_jaccard.py`)
**Output:** `results/guide_outlier_jaccard/guide_outlier_jaccard_heatmap.pdf`

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/cellar/users/aklie/projects/tf_perturb_seq")
sys.path.append(str(PROJECT_ROOT / "config"))
from loader import load_colors

EDIST = PROJECT_ROOT / "docs/jamborees/2026_UTSW/working_groups/wg1_data_qc/edist"
RESULTS = EDIST / "results" / "guide_outlier_jaccard"
RESULTS.mkdir(parents=True, exist_ok=True)

dataset_colors = load_colors("production_TF-Perturb-seq", "dataset_colors")

SHORT = {
    "Hon_WTC11-cardiomyocyte-differentiation_TF-Perturb-seq": "HonCM",
    "Huangfu_HUES8-definitive-endoderm-differentiation_TF-Perturb-seq": "HuangfuDE",
    "Huangfu_HUES8-embryonic-stemcell-differentiation_TF-Perturb-seq": "HuangfuESC",
    "Gersbach_WTC11-hepatocyte-differentiation_TF-Perturb-seq": "GersbachHep",
    "Engreitz_WTC11-endothelial-cells_TF-Perturb-seq": "EngreitzEndo",
}
SHORT_TO_FULL = {v: k for k, v in SHORT.items()}
DATASET_ORDER = ["HonCM", "HuangfuDE", "HuangfuESC", "GersbachHep"]
COLORS = {s: dataset_colors[SHORT_TO_FULL[s]] for s in DATASET_ORDER}

print("RESULTS:", RESULTS)


In [ ]:
jac = pd.read_csv(RESULTS / "guide_outlier_jaccard.tsv", sep="\t", index_col=0)
sizes = jac[["n_total_grnas", "n_outlier_grnas"]]
M = jac.drop(columns=["n_total_grnas", "n_outlier_grnas"]).reindex(
    index=DATASET_ORDER, columns=DATASET_ORDER,
).astype(float)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(M.values, cmap="viridis", vmin=0, vmax=max(M.values[~np.eye(len(M), dtype=bool)].max(), 0.05))
ax.set_xticks(range(len(M))); ax.set_xticklabels(M.columns, rotation=45, ha="right")
ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
for i in range(len(M)):
    for j in range(len(M)):
        v = M.iloc[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v < 0.5 else "black", fontsize=9)

ax.set_title("Jaccard — outlier gRNA sets (pval_outlier < 0.05)")
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
plt.tight_layout()
fig.savefig(RESULTS / "guide_outlier_jaccard_heatmap.pdf")
plt.show()

print()
print("Outlier set sizes:")
print(sizes.to_string())